# SOCOFing robustness study — full reproducible run

Self-contained Kaggle notebook. Re-run top-to-bottom any time (cached gallery +
per-probe resume make re-runs cheap).

- **Experiments** — both evaluation tiers:
  - **Tier A** (fair cross-model): all 3 models on the 2000-probe manifest
    (SIFT on a nested 500).
  - **Tier B** (best estimate): Gabor + DINOv2 on the FULL probe set (SIFT is
    excluded — O(N×gallery)).
- **Analysis** — tables, figures, and paired significance (McNemar + bootstrap).

**Before running:** attach the SOCOFing dataset and enable **Internet**
(Settings → Internet ON) for DINOv2 weights. Tier B is heavy (~hours) but
resume-safe — skip its cell if you only need Tier A.

## Setup

In [ ]:
import os
if not os.path.exists('/kaggle/working/Biometric'):
    !git clone https://github.com/AndreaGiurissich/Biometric.git /kaggle/working/Biometric
%cd /kaggle/working/Biometric
!git pull origin main
!pip install -q -r requirements.txt

In [ ]:
!python scripts/verify_dataset.py --input-root /kaggle/input

## Experiments — Tier A (fair cross-model)

All 3 models × 3 levels × 2 conditions on the 2000-probe manifest. SIFT scoring
is parallelized (`--workers 4`). Resume-safe.

In [ ]:
!python scripts/run_all.py --workers 4 --no-synthesize --no-significance

## Experiments — Tier B (best estimate, full probe set)

Gabor + DINOv2 on EVERY probe (`--full`). Heavy (~hours); results go to separate
`*_full` dirs so they never collide with Tier A. Skip if short on time.

In [ ]:
!python scripts/run_all.py --models gabor,dinov2 --full --workers 4 \
    --no-synthesize --no-significance

## Analysis

Tables + figures + paired significance. Handles both tiers automatically (CSVs
carry a `tier` column; cross-model figures use Tier A).

In [ ]:
!python scripts/synthesize.py        # results/summary.csv, per_alteration.csv, figures/
!python scripts/significance.py      # results/significance.csv (McNemar + bootstrap)

In [ ]:
import glob
from IPython.display import Image, display
for p in sorted(glob.glob('/kaggle/working/results/figures/*.png')):
    print(p); display(Image(p))

In [ ]:
import pandas as pd
print('== overall (tier column) =='); display(pd.read_csv('/kaggle/working/results/summary.csv'))
print('== significance =='); display(pd.read_csv('/kaggle/working/results/significance.csv'))

In [ ]:
# Bundle results for download (Kaggle disk is ephemeral).
!python scripts/save_results.py